In [ ]:
#Final

In [ ]:
# CELL 1 - Install deps
!pip install --quiet pandas numpy requests plotly==5.14.1 folium xgboost==1.7.5 scikit-learn joblib shap matplotlib seaborn statsmodels pyproj


In [ ]:
# CELL 2 - Config
OPENWEATHER_API_KEY = __import__("os").environ.get("OPENWEATHER_API_KEY", "")
CITIES = ["Mumbai", "Delhi", "Bengaluru", "Chennai", "Kolkata"]
OUT_DIR = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
MASTER_CSV = f"{OUT_DIR}/master_weather.csv"
print("Config ok. Cities:", CITIES)


In [ ]:
# CELL 3 - Imports & helpers
import os, io, time, json
from datetime import datetime, timezone
import requests
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
import folium
from folium.plugins import HeatMap
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib
import shap
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import grangercausalitytests

pd.options.display.max_columns = 200


In [ ]:
# CELL 4 - Fetch functions (geocoding, current, forecast, air pollution, UV)
BASE_OW = "https://api.openweathermap.org/data/2.5"

def get_city_coords(city, key):
    url = "https://api.openweathermap.org/geo/1.0/direct"
    params = {"q": city, "limit": 1, "appid": key}
    try:
        r = requests.get(url, params=params, timeout=10); r.raise_for_status()
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception as e:
        print("Geocode error", city, e)
    return None, None

def fetch_current_weather_city(city, key):
    url = f"{BASE_OW}/weather"
    params = {"q": city, "appid": key, "units": "metric"}
    r = requests.get(url, params=params, timeout=12)
    r.raise_for_status()
    j = r.json()
    return {
        "city": city,
        "lat": j["coord"]["lat"],
        "lon": j["coord"]["lon"],
        "temp": j["main"]["temp"],
        "humidity": j["main"]["humidity"],
        "pressure": j["main"]["pressure"],
        "wind_speed": j["wind"].get("speed", np.nan),
        "weather": j["weather"][0]["description"],
        "timestamp_utc": datetime.fromtimestamp(j["dt"], timezone.utc)
    }

def fetch_forecast_city(city, key):
    url = f"{BASE_OW}/forecast"
    params = {"q": city, "appid": key, "units": "metric"}
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    j = r.json()
    rows = []
    lat = j["city"]["coord"]["lat"]; lon = j["city"]["coord"]["lon"]
    for it in j["list"]:
        rows.append({
            "city": city,
            "lat": lat, "lon": lon,
            "timestamp_utc": datetime.fromtimestamp(it["dt"], timezone.utc),
            "temp_f": it["main"]["temp"],
            "humidity_f": it["main"]["humidity"],
            "pressure_f": it["main"]["pressure"],
            "wind_speed_f": it["wind"].get("speed", np.nan),
            "clouds_f": it["clouds"].get("all", np.nan)
        })
    return pd.DataFrame(rows)

def fetch_air_pollution(lat, lon, key):
    url = f"{BASE_OW}/air_pollution"
    params = {"lat": lat, "lon": lon, "appid": key}
    r = requests.get(url, params=params, timeout=12)
    r.raise_for_status()
    j = r.json()
    comp = j["list"][0]["components"]
    return {
        "lat": lat, "lon": lon,
        "co": comp.get("co"), "no": comp.get("no"), "no2": comp.get("no2"),
        "o3": comp.get("o3"), "so2": comp.get("so2"), "pm2_5": comp.get("pm2_5"),
        "pm10": comp.get("pm10"), "nh3": comp.get("nh3"),
        "aqi": j["list"][0]["main"].get("aqi"),
        "timestamp_utc": datetime.fromtimestamp(j["list"][0]["dt"], timezone.utc)
    }

def fetch_uv_index(lat, lon, key):
    url = f"{BASE_OW}/uvi"
    params = {"lat": lat, "lon": lon, "appid": key}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        j = r.json()
        # j may be dict with 'value' or list depending on endpoint; handle both
        if isinstance(j, list) and len(j)>0:
            val = j[0].get("value")
            t = j[0].get("date_iso") or j[0].get("date")
        else:
            val = j.get("value") or j.get("uvi")
            t = j.get("date_iso") or j.get("date")
        return {"lat": lat, "lon": lon, "uv_index": val, "timestamp_utc": datetime.now(timezone.utc)}
    except Exception as e:
        # not fatal — UV may not be present
        print("UV fetch error:", e)
        return {}


In [ ]:
# CELL 5 - Fetch and append forecast rows into master (adds many rows per city)
rows_added = 0
forecast_list = []

for city in CITIES:
    print("Collecting for", city)
    lat, lon = get_city_coords(city, OPENWEATHER_API_KEY)
    if lat is None:
        print("  skip - no coords")
        continue
    # forecast (~40 rows per city)
    try:
        fc = fetch_forecast_city(city, OPENWEATHER_API_KEY)
    except Exception as e:
        print("  forecast fail:", e)
        continue
    # for each forecast timestamp, attach nearest air pollution snapshot (current-like)
    ap = None
    try:
        ap = fetch_air_pollution(lat, lon, OPENWEATHER_API_KEY)
    except Exception as e:
        print("  air poll fail:", e)
    uv = {}
    try:
        uv = fetch_uv_index(lat, lon, OPENWEATHER_API_KEY)
    except:
        uv = {}

    # combine rows
    for _, r in fc.iterrows():
        rec = {
            "city": city,
            "lat": r["lat"], "lon": r["lon"],
            "timestamp_utc": r["timestamp_utc"],
            "temp": r["temp_f"], "humidity": r["humidity_f"], "pressure": r["pressure_f"],
            "wind_speed": r["wind_speed_f"], "clouds": r["clouds_f"]
        }
        if ap:
            rec.update({k: ap.get(k) for k in ["co","no","no2","o3","so2","pm2_5","pm10","nh3","aqi"]})
            rec["ap_timestamp"] = ap.get("timestamp_utc")
        else:
            for k in ["co","no","no2","o3","so2","pm2_5","pm10","nh3","aqi"]:
                rec[k] = np.nan
            rec["ap_timestamp"] = None
        rec.update({"uv_index": uv.get("uv_index", np.nan)})
        forecast_list.append(rec)

# build DataFrame
df_fc_all = pd.DataFrame(forecast_list)
print("Forecast rows:", len(df_fc_all))

# append to master CSV
if os.path.exists(MASTER_CSV):
    df_master = pd.read_csv(MASTER_CSV, parse_dates=["timestamp_utc","ap_timestamp"])
    df_comb = pd.concat([df_master, df_fc_all], ignore_index=True)
    # drop duplicates on city+timestamp
    df_comb['timestamp_utc'] = pd.to_datetime(df_comb['timestamp_utc'])
    df_comb = df_comb.drop_duplicates(subset=['city','timestamp_utc'])
else:
    df_comb = df_fc_all.copy()
df_comb.to_csv(MASTER_CSV, index=False)
print("Master saved:", MASTER_CSV, "rows:", len(df_comb))
df_comb.head()


In [ ]:
# CELL 6 - Preprocess master -> engineered features
df = pd.read_csv(MASTER_CSV, parse_dates=["timestamp_utc","ap_timestamp"])
# ensure pollutant numeric
for col in ["pm2_5","pm10","no2","o3","so2","co","nh3","aqi"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# temporal features
df = df.sort_values(["city","timestamp_utc"])
df["hour"] = df["timestamp_utc"].dt.hour
df["dayofweek"] = df["timestamp_utc"].dt.dayofweek
df["date"] = df["timestamp_utc"].dt.date

# lags & rolling
grouped = df.groupby("city")
df["pm25_lag1"] = grouped["pm2_5"].shift(1)
df["pm25_ma3"] = grouped["pm2_5"].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
df["temp_ma3"] = grouped["temp"].rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)

# standardized AQI proxy (if aqi not present)
df["pm25_proxy"] = df["pm2_5"]  # you can replace with EPA/CPCB conversion to AQI

# save processed
PROC_CSV = f"{OUT_DIR}/processed_master.csv"
df.to_csv(PROC_CSV, index=False)
print("Processed saved to", PROC_CSV)
df.head()


In [ ]:
# CELL 7 - EDA: distributions, seasonal boxplots, correlation matrix
display(df[['city','timestamp_utc','pm2_5','pm10','no2','o3','temp']].head())

# distribution of PM2.5 by city
fig = px.box(df, x="city", y="pm2_5", title="PM2.5 distribution by city (forecast rows)")
fig.show()

# diurnal pattern: mean PM2.5 by hour per city
diurnal = df.groupby(['city','hour'])['pm2_5'].mean().reset_index()
fig2 = px.line(diurnal, x='hour', y='pm2_5', color='city', title='Mean diurnal PM2.5 by city')
fig2.show()

# correlation heatmap (numeric)
numcols = ["pm2_5","pm10","no2","o3","so2","co","temp","humidity","wind_speed","uv_index"]
numcols = [c for c in numcols if c in df.columns]
corr = df[numcols].corr()
plt.figure(figsize=(10,8)); sns.heatmap(corr, annot=True, cmap="Spectral"); plt.title("Correlation matrix")
plt.show()


In [ ]:
# CELL 8 - STL decomposition for pm2_5 (per city)
for city in df['city'].unique():
    city_df = df[df['city']==city].sort_values('timestamp_utc').set_index('timestamp_utc')
    if 'pm2_5' not in city_df.columns or city_df['pm2_5'].dropna().shape[0] < 24:
        print("skip STL for", city, "insufficient data")
        continue
    # resample hourly (forecast already hourly/3-hourly); use forward fill
    ts = city_df['pm2_5'].resample('H').mean().interpolate()
    stl = STL(ts, period=24, robust=True).fit()
    fig = stl.plot()
    fig.suptitle(f"STL decomposition PM2.5 — {city}")
    plt.show()


In [ ]:
# CELL 10 - IDW interpolation using city points -> heatmap (coarse)
latest = df.sort_values('timestamp_utc').groupby('city').tail(1).dropna(subset=['lat','lon','pm2_5'])
if latest.shape[0] >= 2:
    # create grid
    lats = np.linspace(latest['lat'].min()-0.5, latest['lat'].max()+0.5, 200)
    lons = np.linspace(latest['lon'].min()-0.5, latest['lon'].max()+0.5, 200)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    # inverse distance weighting
    pts = np.array(list(zip(latest['lat'], latest['lon'])))
    vals = latest['pm2_5'].fillna(0).values
    def idw(xy, pts, vals, power=2):
        dist = np.sqrt(((pts - xy)**2).sum(axis=1))
        # avoid zeros
        if np.any(dist==0):
            return vals[dist.argmax()]  # exact location
        w = 1/(dist**power)
        return (w * vals).sum() / w.sum()
    grid_vals = np.vectorize(lambda y,x: idw(np.array([y,x]), pts, vals))(lat_grid, lon_grid)
    # make heatmap points (coarsen to reduce size)
    heat_points = []
    step = 8
    for i in range(0, grid_vals.shape[0], step):
        for j in range(0, grid_vals.shape[1], step):
            heat_points.append([lat_grid[i,j], lon_grid[i,j], float(grid_vals[i,j])])
    # folium map center
    center = [latest['lat'].mean(), latest['lon'].mean()]
    m = folium.Map(location=center, zoom_start=5)
    HeatMap(heat_points, radius=25, blur=15, max_zoom=10).add_to(m)
    for _, r in latest.iterrows():
        folium.Marker([r['lat'], r['lon']], popup=f"{r['city']} PM2.5 {r['pm2_5']:.1f}").add_to(m)
    map_path = f"{OUT_DIR}/pm25_idw_map.html"
    m.save(map_path)
    print("IDW heatmap saved to", map_path)
else:
    print("Not enough city points for IDW heatmap")


In [ ]:
# CELL 11 - PCA & clustering
cols = [c for c in ['pm2_5','pm10','no2','o3','so2','co'] if c in df.columns]
agg = df.groupby('city')[cols].mean().dropna()
if agg.shape[0] >= 2:
    scaler = StandardScaler()
    Xs = scaler.fit_transform(agg)
    pca = PCA(n_components=min(3, Xs.shape[1]))
    Xp = pca.fit_transform(Xs)
    print("Explained variance ratios:", pca.explained_variance_ratio_)
    kmeans = KMeans(n_clusters=min(4, len(agg)), random_state=42).fit(Xp)
    agg['cluster'] = kmeans.labels_
    display(agg.reset_index())
    fig = px.scatter_3d(
        x=Xp[:,0], y=Xp[:,1], z= Xp[:,2] if Xp.shape[1]>2 else Xp[:,1],
        color=agg['cluster'].astype(str), hover_name=agg.index, title="PCA of city pollutant profiles"
    )
    fig.show()
else:
    print("Not enough cities for PCA/clustering")


In [ ]:
# CELL 12 - Granger causality: does weather -> pm2_5?
results = {}
for city in df['city'].unique():
    sub = df[df['city']==city].sort_values('timestamp_utc').set_index('timestamp_utc')
    if 'pm2_5' not in sub.columns or sub['pm2_5'].dropna().shape[0] < 30:
        continue
    # prepare series (resample hourly)
    tmp = sub[['pm2_5','temp','humidity','wind_speed']].resample('H').mean().interpolate()
    for var in ['temp','humidity','wind_speed']:
        try:
            test = grangercausalitytests(tmp[['pm2_5', var]].dropna(), maxlag=6, verbose=False)
            # collect p-value for lag 1
            pval = test[1][0]['ssr_ftest'][1]
            results[(city,var)] = pval
        except Exception as e:
            results[(city,var)] = None
results


In [ ]:
# CELL 13 - Forecasting with XGBoost + SHAP explainability (per-city)
models_info = {}
for city in df['city'].unique():
    sub = df[df['city']==city].dropna(subset=['pm2_5']).sort_values('timestamp_utc')
    if len(sub) < 50:
        print("skip ML for", city, "insufficient rows:", len(sub)); continue
    feats = [c for c in ['temp','humidity','pressure','wind_speed','pm10','no2','o3','uv_index'] if c in sub.columns]
    sub = sub.dropna(subset=feats + ['pm2_5'])
    X = sub[feats]; y = sub['pm2_5']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
    model.fit(X_train, y_train, eval_set=[(X_test,y_test)], early_stopping_rounds=25, verbose=False)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds) if len(y_test)>1 else np.nan
    models_info[city] = {"model": model, "mae": mae, "r2": r2, "features": feats}
    print(f"{city} MAE {mae:.2f} R2 {r2}")
    # SHAP explain last test sample
    try:
        expl = shap.Explainer(model)
        sample = X_test.tail(1)
        sv = expl(sample)
        print("SHAP values for last sample:")
        display(pd.DataFrame({"feature": sample.columns, "value": sample.iloc[0].values, "shap": sv.values[0]}))
    except Exception as e:
        print("SHAP fail:", e)
    # save model
    joblib.dump(model, f"{OUT_DIR}/xgb_{city}.pkl")


In [ ]:
# This base code will be needed when  using sensors-Real-Time NASA FIRMS Fire Detection for Specific Area ---

import requests
import pandas as pd
import io
from math import radians, sin, cos, sqrt, atan2
import json

# --------------------------------------------
# 🌍 Helper Function: Calculate distance
# --------------------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Radius of Earth in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# --------------------------------------------
# 🔑 NASA FIRMS AREA API CONFIG
# --------------------------------------------
MAP_KEY = __import__("os").environ.get("NASA_FIRMS_KEY", "")

# VIIRS_SNPP_NRT = Suomi NPP satellite, near-real-time
SENSOR = "VIIRS_SNPP_NRT"

# Bounding box for region (lat_min, lon_min, lat_max, lon_max)
# Example: around Mumbai region (~100km area)
AREA_BOX = "74.0,30.3,75.3,31.35"

# Number of days (1 = past 24h)
DAYS = "1"

# Build API URL
NASA_API_URL = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SENSOR}/{AREA_BOX}/{DAYS}"

print("🌐 Requesting NASA FIRMS area data from:")
print(NASA_API_URL)

# --------------------------------------------
# 📡 Fetch and Parse
# --------------------------------------------
response = requests.get(NASA_API_URL, timeout=60)

if response.status_code != 200:
    raise Exception(f"FIRMS API Error: {response.status_code} - {response.text[:200]}")

fire_df = pd.read_csv(io.StringIO(response.text))
print(f"🔥 {len(fire_df)} active fires detected in your area")

# Identify correct column names dynamically
lat_col = [c for c in fire_df.columns if 'lat' in c.lower()][0]
lon_col = [c for c in fire_df.columns if 'lon' in c.lower()][0]

# --------------------------------------------
# 📍 Example Sensor Data (replace with live readings)
# --------------------------------------------
sensor_data = {
    "latitude": 19.0760,
    "longitude": 72.8777,
    "PM2_5": 153.2,
    "Gas": 0.58,
    "Temperature": 31.4,
    "Humidity": 65.2
}

# --------------------------------------------
# 🔍 Filter fires within 25 km radius
# --------------------------------------------
nearby_fires = []
for _, row in fire_df.iterrows():
    dist = haversine(sensor_data["latitude"], sensor_data["longitude"], row[lat_col], row[lon_col])
    if dist <= 100:
        nearby_fires.append({
            "latitude": row[lat_col],
            "longitude": row[lon_col],
            "confidence": row.get("confidence", None),
            "bright_ti4": row.get("bright_ti4", None),
            "distance_km": round(dist, 2)
        })

# --------------------------------------------
# 📊 Combine and Display Result
# --------------------------------------------
result = {
    "Sensor_Data": sensor_data,
    "Nearby_Fires": nearby_fires,
    "Fire_Count": len(nearby_fires),
    "Advisory": "🔥 Fire detected nearby – possible pollution source"
                if len(nearby_fires) > 0
                else "✅ No nearby fire sources detected"
}

print(json.dumps(result, indent=4))


In [ ]:
#Final- Real-Time NASA FIRMS Fire Detection for Fixed AREA_BOX

import requests
import pandas as pd
import io
import json

# --------------------------------------------
# 🔑 NASA FIRMS AREA API CONFIG
# --------------------------------------------
MAP_KEY = __import__("os").environ.get("NASA_FIRMS_KEY", "")
SENSOR = "VIIRS_SNPP_NRT"

# --------------------------------------------
# 🗺️ Fixed Bounding Box
# --------------------------------------------
AREA_BOX = "74.0,30.3,75.3,31.35"   # lat_min, lon_min, lat_max, lon_max
DAYS = "1"  # past 24 hours

NASA_API_URL = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SENSOR}/{AREA_BOX}/{DAYS}"
print("🌐 Requesting NASA FIRMS area data from:")
print(NASA_API_URL)

# --------------------------------------------
# 📡 Fetch and Parse FIRMS data
# --------------------------------------------
try:
    response = requests.get(NASA_API_URL, timeout=60)
    response.raise_for_status()
    fire_df = pd.read_csv(io.StringIO(response.text))
    print(f" {len(fire_df)} active fires detected in bounding box")
except:
    # If API fails, use simulated data for demonstration
    print("⚠️ NASA FIRMS API not available – using simulated fire data")
    fire_df = pd.DataFrame({
        'latitude': [30.5, 30.8, 31.0],
        'longitude': [74.5, 74.9, 75.0],
        'confidence': ['nominal', 'high', 'low'],
        'bright_ti4': [320, 330, 310]
    })

# Identify latitude and longitude columns dynamically
lat_col = [c for c in fire_df.columns if 'lat' in c.lower()][0]
lon_col = [c for c in fire_df.columns if 'lon' in c.lower()][0]

# --------------------------------------------
# 🔍 Capture all fires within AREA_BOX
# --------------------------------------------
# Correctly parse AREA_BOX
lon_min, lat_min, lon_max, lat_max = map(float, AREA_BOX.split(','))
fires_in_box = []

for _, row in fire_df.iterrows():
    if lat_min <= row[lat_col] <= lat_max and lon_min <= row[lon_col] <= lon_max:
        fires_in_box.append({
            "latitude": row[lat_col],
            "longitude": row[lon_col],
            "confidence": row.get("confidence", None),
            "bright_ti4": row.get("bright_ti4", None)
        })

# --------------------------------------------
# 📊 Display Result
# --------------------------------------------
result = {
    "Bounding_Box": AREA_BOX,
    "Nearby_Fires": fires_in_box,
    "Fire_Count": len(fires_in_box),
    "Advisory": "Fires detected in bounding box – possible pollution source"
                if len(fires_in_box) > 0
                else "No fires detected in bounding box"
}

print(json.dumps(result, indent=4))


In [ ]:
import requests, math, time, re, json
from datetime import datetime
from IPython.display import display, Markdown

WAQI_TOKEN = __import__("os").environ.get("WAQI_TOKEN", "")
TOMTOM_KEY = __import__("os").environ.get("TOMTOM_KEY", "")
RTDMS_TOKEN = __import__("os").environ.get("RTDMS_TOKEN", "")

CPCB_ENDPOINTS = {
    "mumbai":    {"source":"SAFAR (Mumbai)", "url":"https://safar.tropmet.res.in/map_data.php?city_id=2&for=current"},
    "pune":      {"source":"SAFAR (Pune)",   "url":"https://safar.tropmet.res.in/map_data.php?city_id=1&for=current"},
    "delhi":     {"source":"SAFAR (Delhi)",  "url":"https://safar.tropmet.res.in/map_data.php?city_id=3&for=current"},
    "ahmedabad": {"source":"SAFAR (Ahmedabad)","url":"https://safar.tropmet.res.in/map_data.php?city_id=4&for=current"},
    "cpcb_rtdms_template": {"source":"CPCB RTDMS (requires token & station id)","url":"https://rtdms.cpcb.gov.in/v1.0/station/{station_id}/latest"}
}

def normalize(s): return re.sub(r'[^a-z0-9 ]',' ', (s or '').lower()).strip()
def haversine(lat1, lon1, lat2, lon2):
    R=6371.0
    phi1,phi2=math.radians(lat1),math.radians(lat2)
    dphi=math.radians(lat2-lat1); dl=math.radians(lon2-lon1)
    a=math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def pm25_to_aqi(pm25):
    if pm25 is None: return None
    C=float(pm25)
    bp=[(0.0,12.0,0,50),(12.1,35.4,51,100),(35.5,55.4,101,150),(55.5,150.4,151,200),(150.5,250.4,201,300),(250.5,350.4,301,400),(350.5,500.4,401,500)]
    for lo,hi,ilo,ihi in bp:
        if lo<=C<=hi:
            return round((ihi-ilo)/(hi-lo)*(C-lo)+ilo)
    return 500 if C>500.4 else None

def interpret_aqi(aqi):
    if aqi is None: return "Unknown"
    aqi=int(aqi)
    if aqi<=50: return "🟢 Good (0–50)"
    if aqi<=100: return "🟡 Moderate (51–100)"
    if aqi<=150: return "🟠 Unhealthy for Sensitive Groups (101–150)"
    if aqi<=200: return "🔴 Unhealthy (151–200)"
    if aqi<=300: return "🟣 Very Unhealthy (201–300)"
    return "⚫ Hazardous (301+)"

def classify_cause(meas):
    causes=[]
    pm25=meas.get('pm25'); pm10=meas.get('pm10'); no2=meas.get('no2'); so2=meas.get('so2'); o3=meas.get('o3'); co=meas.get('co')
    if pm25 and pm25>60: causes.append("High PM2.5 — vehicles/biomass burning/secondary aerosols")
    elif pm25 and pm25>35: causes.append("Elevated PM2.5 — vehicular emissions & combustion")
    if pm10 and pm10>100: causes.append("High PM10 — road dust / construction")
    if no2 and no2>80: causes.append("High NO₂ — vehicle emissions")
    if so2 and so2>80: causes.append("High SO₂ — industrial/coal combustion")
    if o3 and o3>100: causes.append("High O₃ — photochemical smog")
    if co and co>5: causes.append("High CO — incomplete combustion")
    if not causes: causes.append("Mixed urban sources (traffic + dust)")
    return " & ".join(causes)

display(Markdown("### 🇮🇳 Live AQI + Traffic (single-run)"))
place_in = input().strip()
age_group = input().strip().title()
health_condition = input().strip().title()
strict_mode = input().strip().lower() == 'y'

coord_m = re.match(r'^\s*([-+]?\d+(\.\d+)?)[,\s]+([-+]?\d+(\.\d+)?)\s*$', place_in)
if coord_m:
    lat=float(coord_m.group(1)); lon=float(coord_m.group(3))
    display_name=f"Coordinates {lat:.6f},{lon:.6f}"
else:
    try:
        g = requests.get(f"https://geocoding-api.open-meteo.com/v1/search?name={requests.utils.quote(place_in)}&count=10&language=en&country=IN", timeout=12).json()
    except Exception as e:
        raise SystemExit(f"Geocoding failed: {e}")
    if "results" not in g or len(g["results"])==0:
        raise SystemExit("Could not geocode within India.")
    selected = g["results"][0]
    token = normalize(place_in)
    for r in g["results"]:
        name = (r.get("name") or "") + " " + (r.get("admin1") or "")
        if token and token in normalize(name):
            selected = r; break
    if token not in normalize(selected.get("name","")):
        for r in g["results"]:
            if "maharashtra" in (r.get("admin1") or "").lower():
                selected = r; break
    lat = float(selected["latitude"]); lon = float(selected["longitude"])
    display_name = f"{selected.get('name','')}, {selected.get('admin1','')}".strip(", ")

openaq_url = f"https://api.openaq.org/v2/locations?coordinates={lat},{lon}&radius=50000&country=IN&order_by=distance&limit=50"
try:
    openaq_j = requests.get(openaq_url, timeout=12).json()
    openaq_locs = openaq_j.get('results', []) if isinstance(openaq_j, dict) else []
except:
    openaq_locs=[]

pref_tokens=set()
for t in re.split(r'[\s,]+', normalize(display_name)):
    if t: pref_tokens.add(t)

def station_matches_tokens(st):
    city = normalize(st.get('city') or "")
    nm = normalize(st.get('name') or "")
    combined = city + " " + nm
    matches = sum(1 for tk in pref_tokens if tk and tk in combined)
    return matches > 0

same_city=[]; nearby=[]
for st in openaq_locs:
    coords=st.get('coordinates') or {}
    if not coords: continue
    st_lat=coords.get('latitude'); st_lon=coords.get('longitude')
    if st_lat is None or st_lon is None: continue
    dist = haversine(lat, lon, st_lat, st_lon)
    item={'id':st.get('id'),'name':st.get('name'),'city':st.get('city'),'lat':st_lat,'lon':st_lon,'dist':dist,'lastUpdated':st.get('lastUpdated')}
    if station_matches_tokens(st): same_city.append(item)
    else: nearby.append(item)
same_city=sorted(same_city, key=lambda x:x['dist'])
nearby=sorted(nearby, key=lambda x:x['dist'])

chosen_station=None; chosen_source=""; reliability=""
if same_city:
    chosen_station=same_city[0]; chosen_source="OpenAQ (same-city station)"; reliability="High (same-city monitor)"
elif nearby and not strict_mode:
    chosen_station=nearby[0]; chosen_source="OpenAQ (nearest station fallback)"; reliability="Medium (nearest monitor)"
else:
    chosen_station=None

cpcb_used=False; cpcb_raw=None
city_key = normalize(display_name).split()[0] if display_name else ""
if city_key in CPCB_ENDPOINTS and CPCB_ENDPOINTS[city_key].get('url'):
    ep=CPCB_ENDPOINTS[city_key]
    try:
        r = requests.get(ep['url'], timeout=8)
        if r.status_code==200:
            try: cpcb_raw = r.json()
            except: cpcb_raw = {'text': r.text}
            chosen_source = f"{ep['source']}"
            reliability = "High (official SAFAR endpoint)"
            cpcb_used=True
    except:
        cpcb_used=False

measurements={'pm25':None,'pm10':None,'no2':None,'so2':None,'o3':None,'co':None}
if chosen_station:
    try:
        latest_url = f"https://api.openaq.org/v2/latest?location_id={chosen_station['id']}&limit=100"
        lj = requests.get(latest_url, timeout=10).json()
        res = lj.get('results',[])
        if res:
            for m in res[0].get('measurements',[]):
                p=m.get('parameter'); v=m.get('value')
                if p=='pm25': measurements['pm25']=v
                if p=='pm10': measurements['pm10']=v
                if p=='no2': measurements['no2']=v
                if p=='so2': measurements['so2']=v
                if p=='o3': measurements['o3']=v
                if p=='co': measurements['co']=v
    except:
        pass

if cpcb_used and isinstance(cpcb_raw, dict):
    def deep_find_pm25(obj):
        if isinstance(obj, dict):
            for k,v in obj.items():
                if isinstance(k,str) and 'pm' in k.lower() and '25' in k.lower():
                    try: return float(v)
                    except: pass
                found = deep_find_pm25(v)
                if found is not None: return found
        if isinstance(obj, list):
            for it in obj:
                found = deep_find_pm25(it)
                if found is not None: return found
        return None
    cp_pm25 = deep_find_pm25(cpcb_raw)
    if cp_pm25 is not None:
        measurements['pm25'] = cp_pm25

waqi_ok=False; waqi_info={}
try:
    waqi_j = requests.get(f"https://api.waqi.info/feed/geo:{lat};{lon}/?token={WAQI_TOKEN}", timeout=10).json()
    if waqi_j.get('status')=='ok':
        waqi_ok=True
        waqi_info['aqi']=waqi_j['data'].get('aqi')
        waqi_info['city']=waqi_j['data'].get('city',{}).get('name')
        waqi_info['dominant']=waqi_j['data'].get('dominentpol')
        waqi_info['time']=waqi_j['data'].get('time',{}).get('s')
        if not chosen_station and (not strict_mode or (normalize(waqi_info['city']).find(normalize(display_name))!=-1)):
            chosen_source=f"WAQI ({waqi_info.get('city')})"; reliability="Medium (WAQI station)"
except:
    waqi_ok=False

om_pm25=None; om_no2=None; om_time=None
try:
    om_url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&hourly=pm2_5,pm10,no2,so2,o3,co&timezone=auto"
    omj = requests.get(om_url, timeout=10).json()
    times = omj.get('hourly',{}).get('time',[])
    if times:
        idx=len(times)-1
        om_pm25 = omj.get('hourly',{}).get('pm2_5',[])[idx] if omj.get('hourly',{}).get('pm2_5') else None
        om_no2  = omj.get('hourly',{}).get('no2',[])[idx] if omj.get('hourly',{}).get('no2') else None
        om_time = times[idx]
        if measurements['pm25'] is None: measurements['pm25'] = om_pm25
        if measurements['no2'] is None: measurements['no2'] = om_no2
except:
    pass

rtdms_used=False; rtdms_raw=None

primary_source=None; primary_aqi=None; primary_ts=None; station_distance=None
if cpcb_used:
    primary_source=chosen_source
    if measurements.get('pm25') is not None:
        primary_aqi = pm25_to_aqi(measurements.get('pm25'))
elif chosen_station:
    primary_source=chosen_source
    if measurements.get('pm25') is not None:
        primary_aqi = pm25_to_aqi(measurements.get('pm25'))
    primary_ts = chosen_station.get('lastUpdated')
    station_distance = chosen_station.get('dist')
elif waqi_ok and waqi_info.get('aqi') is not None:
    primary_source=chosen_source
    try: primary_aqi = int(waqi_info.get('aqi'))
    except: primary_aqi=None
    primary_ts = waqi_info.get('time')
else:
    primary_source="Open-Meteo grid (fallback)"
    if measurements.get('pm25') is not None:
        primary_aqi = pm25_to_aqi(measurements.get('pm25'))
    primary_ts = om_time

hour_now = datetime.now().hour
try:
    city_guess = display_name
    large = any(x in normalize(city_guess) for x in ['mumbai','delhi','bangalore','bengaluru','chennai','kolkata','hyderabad','pune'])
    no2_val = measurements.get('no2') or 0
    rush = 1 if (7 <= hour_now <= 11 or 17 <= hour_now <= 21) else 0
    score = (2 if large else 0) + 2*rush + (1 if no2_val>60 else 0)
    if score>=4: heuristic="Heavy Traffic (heuristic)"
    elif score>=2: heuristic="Moderate Traffic (heuristic)"
    else: heuristic="Light Traffic (heuristic)"
except:
    heuristic="Heuristic unavailable"

tomtom_cond=None; tomtom_info={}
if TOMTOM_KEY:
    try:
        turl = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?point={lat},{lon}&key={TOMTOM_KEY}"
        tj = requests.get(turl, timeout=8).json()
        fs = tj.get('flowSegmentData',{})
        cs = fs.get('currentSpeed'); ff = fs.get('freeFlowSpeed')
        if cs and ff:
            r = cs/ff
            if r<0.5: tomtom_cond="Heavy Traffic (TomTom)"
            elif r<0.8: tomtom_cond="Moderate Traffic (TomTom)"
            else: tomtom_cond="Light Traffic (TomTom)"
            tomtom_info={'currentSpeed':cs,'freeFlow':ff}
    except:
        tomtom_cond=None

final_traffic = tomtom_cond if tomtom_cond else heuristic

cause = classify_cause(measurements)
if waqi_ok and waqi_info.get('dominant'): cause += f"  (WAQI dominant: {waqi_info.get('dominant').upper()})"

def who_advice(aqi, age, cond, traffic):
    lines=[]
    if aqi is None:
        lines.append("AQI unknown — be cautious if you are sensitive or have a condition.")
        return lines
    if aqi<=50: lines.append("🟢 Good — normal outdoor activities OK.")
    elif aqi<=100: lines.append("🟡 Moderate — sensitive individuals reduce prolonged exertion.")
    elif aqi<=150: lines.append("🟠 Unhealthy for Sensitive Groups — children/elderly/lung/heart patients limit outdoor time.")
    elif aqi<=200: lines.append("🔴 Unhealthy — everyone may begin to experience health effects; avoid outdoor exertion.")
    elif aqi<=300: lines.append("🟣 Very Unhealthy — stay indoors with filtration; avoid exercise.")
    else: lines.append("⚫ Hazardous — emergency conditions; remain indoors; seek help if symptoms occur.")
    age_l=(age or "").lower(); c=(cond or "").lower()
    if 'child' in age_l: lines.append("• Children: avoid outdoor play when AQI > 100.")
    if 'elder' in age_l: lines.append("• Elderly: avoid stepping out during poor AQI; keep meds handy.")
    if 'asth' in c: lines.append("• Asthmatic: keep inhaler ready; avoid triggers.")
    if 'heart' in c: lines.append("• Heart patient: avoid exertion; consult physician if symptoms.")
    if 'lung' in c: lines.append("• Lung condition: use N95/FFP2 masks outdoors; run indoor HEPA filters.")
    if 'heavy' in (traffic or "").lower() or (aqi is not None and aqi>150):
        lines.append("• ⚠️ Avoid long commutes; use N95/FFP2 mask if travel required.")
    elif 'moderate' in (traffic or "").lower() or (aqi is not None and aqi>100):
        lines.append("• 🟠 Reduce outdoor time during rush hours; prefer public transport.")
    return lines

print("\n================== LIVE AQI & TRAFFIC REPORT (India - Version A) ==================\n")
print(f"Query: {display_name}  (lat: {lat:.6f}, lon: {lon:.6f})")
print(f"Selection mode: {'Strict (same-city only)' if strict_mode else 'Allow nearest-fallback'}")
print("\nPrimary data source:", primary_source)
if station_distance is not None: print(f" • Station distance: {station_distance:.2f} km")
if primary_ts: print(f" • Data timestamp: {primary_ts}")
if waqi_ok and waqi_info.get('city'): print(f" • WAQI station (nearby): {waqi_info.get('city')}, dominant pollutant: {waqi_info.get('dominant')}")


print("\nAQI:")
print(f" • Value: {primary_aqi if primary_aqi is not None else 'N/A'}  — {interpret_aqi(primary_aqi) if primary_aqi is not None else ''}")

print("\nTraffic summary (TomTom + Heuristic):")
print(f" • TomTom condition: {tomtom_cond if tomtom_cond else 'TomTom not available or no segment data'}")
if tomtom_info: print(f"   - TomTom speeds: current {tomtom_info.get('currentSpeed')} km/h, free-flow {tomtom_info.get('freeFlow')} km/h")
print(f" • Heuristic: {heuristic}")
print(f" => Final traffic assessment: {final_traffic}")

print("\nLikely pollution cause(s):")
print(f" • {cause}")

print("\nPersonalized WHO-style health recommendations:")
for line in who_advice(primary_aqi, age_group, health_condition, final_traffic):
    print(" •", line)


